In [1]:
import logging
from pathlib import Path

import numpy as np
import pandas as pd
from joblib import Parallel, delayed
from lmfit.models import LinearModel, LorentzianModel
from matplotlib import colors
from matplotlib import pyplot as plt
from matplotlib_scalebar.scalebar import ScaleBar
from scipy import sparse
from scipy.interpolate import interp1d
from scipy.sparse.linalg import spsolve

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(name)s - %(levelname)s - %(message)s",
)
logger = logging.getLogger(__name__)

DATA_DIR = Path("data")
OUTPUT_DIR = Path("output")

D_PEAK_RANGE = (1280, 1400)
G_PEAK_RANGE = (1500, 1750)
TWO_D_PEAK_RANGE = (2500, 2800)

PLOT_STYLE = {
    "font.family": "Times New Roman",
    "axes.linewidth": 1.0,
    "xtick.direction": "in",
    "ytick.direction": "in",
    "xtick.top": True,
    "ytick.right": True,
}


In [2]:
def load_mapping_ascii(filepath: Path) -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    """2DマッピングASCIIファイルを読み込み (x, y, wavenumber, spectra) を返す"""
    with open(filepath, "r") as f:
        lines = [line.strip().split() for line in f if line.strip()]

    wavenumber = np.array([float(v) for v in lines[0]])
    rows = []
    for parts in lines[1:]:
        try:
            rows.append([float(v) for v in parts])
        except ValueError:
            logger.warning(f"Skipping malformed line in {filepath.name}")

    data = np.array(rows)
    return data[:, 0], data[:, 1], wavenumber, data[:, 2:]


def load_single_ascii(filepath: Path) -> tuple[np.ndarray, np.ndarray]:
    """1D単一スペクトルASCIIファイルを読み込み (wavenumber, intensity) を返す"""
    df = pd.read_csv(filepath, sep=r"\s+", header=None, names=["X", "Y"]).dropna().sort_values("X")
    return df["X"].to_numpy(), df["Y"].to_numpy()


def correct_wavenumber_shift(
    spectra: np.ndarray,
    wavenumber: np.ndarray,
    si_range: tuple[float, float] = (500, 600),
    target: float = 520.7,
) -> tuple[np.ndarray, np.ndarray]:
    """Siピーク位置 (520.7 cm^-1) を基準に各ピクセルの波数シフトを線形補間で補正する"""
    spectra = np.atleast_2d(spectra)
    mask = (wavenumber >= si_range[0]) & (wavenumber <= si_range[1])
    wn_si = wavenumber[mask]

    corrected = np.zeros_like(spectra)
    shifts = np.zeros(len(spectra))

    for i, spec in enumerate(spectra):
        shift = wn_si[np.argmax(spec[mask])] - target
        shifts[i] = shift
        f = interp1d(wavenumber - shift, spec, kind="linear", bounds_error=False, fill_value="extrapolate")  # type: ignore
        corrected[i] = f(wavenumber)

    return corrected, shifts


def baseline_als(y: np.ndarray, lam: float = 1e5, p: float = 0.01, niter: int = 10) -> np.ndarray:
    """Asymmetric Least Squares (ALS) によるベースライン推定"""
    if niter <= 0:
        return y.copy()
    L = len(y)
    D = sparse.diags([1.0, -2.0, 1.0], [0, -1, -2], shape=(L, L - 2), dtype=float)  # type: ignore
    W = sparse.eye(L, dtype=float)
    z = y.copy()
    for _ in range(niter):
        W.setdiag(p * (y > 0) + (1 - p) * (y < 0))
        Z = (W + lam * D @ D.T).tocsc()
        z = spsolve(Z, W @ y)
    return z


def preprocess_baseline(
    spectra: np.ndarray,
    wavenumber: np.ndarray,
    target_range: tuple[float, float] = (1100, 3000),
) -> np.ndarray:
    """指定波数範囲に対してALSベースライン差し引きを行い、範囲外を0にする"""
    spectra = np.atleast_2d(spectra)
    corrected = np.zeros_like(spectra)
    mask = (wavenumber >= target_range[0]) & (wavenumber <= target_range[1])

    for i, spec in enumerate(spectra):
        cropped = spec[mask]
        corrected[i, mask] = cropped - baseline_als(cropped)

    return corrected


In [3]:
def fit_lorentzian_peaks(x: np.ndarray, y: np.ndarray, centers_init: list[float]):
    """直線背景 + 1個以上のローレンツ関数でフィッティングを実行する"""
    model = LinearModel(prefix="bkg_")
    params = model.make_params(slope=0, intercept=np.min(y))

    for i, c_val in enumerate(centers_init):
        prefix = f"p{i}_"
        p_model = LorentzianModel(prefix=prefix)
        model += p_model
        params.update(p_model.make_params())

        params[f"{prefix}center"].set(value=c_val, min=c_val - 20, max=c_val + 20)
        idx = int(np.abs(x - c_val).argmin())
        height_init = y[idx] - np.min(y)
        sigma_init = 15.0
        params[f"{prefix}amplitude"].set(value=height_init * np.pi * sigma_init, min=0)
        params[f"{prefix}sigma"].set(value=sigma_init, min=3, max=100)

    return model.fit(y, params, x=x)


def _fit_single_pixel(i: int, wavenumber: np.ndarray, spectrum: np.ndarray):
    """1ピクセル分のD・G・2Dピークをフィッティングし、定量値辞書とフィット結果オブジェクトを返す"""
    def _init_center(rng: tuple[float, float], default: float) -> float:
        m = (wavenumber >= rng[0]) & (wavenumber <= rng[1])
        return float(wavenumber[m][np.argmax(spectrum[m])]) if np.any(m) else default

    d_init = _init_center(D_PEAK_RANGE, 1350.0)
    g_init = _init_center((G_PEAK_RANGE[0], 1700), 1580.0)
    twod_init = _init_center(TWO_D_PEAK_RANGE, 2680.0)

    try:
        mask_dg = (wavenumber >= D_PEAK_RANGE[0]) & (wavenumber <= G_PEAK_RANGE[1])
        res_dg = fit_lorentzian_peaks(wavenumber[mask_dg], spectrum[mask_dg], [d_init, g_init])

        mask_2d = (wavenumber >= TWO_D_PEAK_RANGE[0]) & (wavenumber <= TWO_D_PEAK_RANGE[1])
        res_2d = fit_lorentzian_peaks(wavenumber[mask_2d], spectrum[mask_2d], [twod_init])

        id_val = float(res_dg.params["p0_height"].value)
        ig_val = float(res_dg.params["p1_height"].value)
        i2d_val = float(res_2d.params["p0_height"].value)

        metrics = {
            "G_Intensity": ig_val,
            "D_Intensity": id_val,
            "2D_Intensity": i2d_val,
            "D_G_Ratio": id_val / (ig_val + 1e-12),
            "2D_G_Ratio": i2d_val / (ig_val + 1e-12),
            "2D_FWHM": float(res_2d.params["p0_fwhm"].value),
            "D_Center": float(res_dg.params["p0_center"].value),
            "G_Center": float(res_dg.params["p1_center"].value),
            "2D_Center": float(res_2d.params["p0_center"].value),
            "2D_RedChi": float(res_2d.redchi),
        }
        return i, metrics, {"dg": res_dg, "2d": res_2d}

    except Exception:
        import traceback
        traceback.print_exc()
        empty = {
            "G_Intensity": 0.0, "D_Intensity": 0.0, "2D_Intensity": 0.0,
            "D_G_Ratio": 0.0, "2D_G_Ratio": 0.0,
            "2D_FWHM": np.nan, "D_Center": np.nan, "G_Center": np.nan,
            "2D_Center": np.nan, "2D_RedChi": np.nan,
        }
        return i, empty, None


def fit_all_spectra(
    wavenumber: np.ndarray,
    spectra: np.ndarray,
    x: np.ndarray | None = None,
    y: np.ndarray | None = None,
) -> tuple[pd.DataFrame, dict[int, dict]]:
    """全ピクセルを並列フィッティングし、結果DataFrameとフィット辞書を返す"""
    spectra = np.atleast_2d(spectra)
    logger.info("Fitting all pixels using joblib multiprocessing...")

    results = Parallel(n_jobs=-1, verbose=5)(
        delayed(_fit_single_pixel)(i, wavenumber, spectra[i]) for i in range(len(spectra))
    )

    rows = []
    fit_results_dict: dict[int, dict] = {}
    for i, metrics, fit_obj in results:  # type: ignore
        row = {}
        if x is not None and y is not None:
            row["X"] = x[i]
            row["Y"] = y[i]
        row.update(metrics)
        rows.append(row)
        if fit_obj is not None:
            fit_results_dict[i] = fit_obj

    return pd.DataFrame(rows), fit_results_dict


In [4]:
def export_single_spectrum_data(
    filepath: Path,
    wavenumber: np.ndarray,
    raw_spectrum: np.ndarray,
    processed_spectrum: np.ndarray,
    fit_results: dict | None = None,
) -> None:
    """1ピクセル分の生波形・処理後波形・フィット曲線をCSV保存する"""
    filepath.parent.mkdir(parents=True, exist_ok=True)
    data = {
        "Wavenumber": wavenumber,
        "Raw_Intensity": raw_spectrum,
        "Processed_Intensity": processed_spectrum,
    }
    if fit_results is not None:
        fit_curve = np.zeros_like(wavenumber, dtype=float)
        for key in ("dg", "2d"):
            res = fit_results.get(key)
            if res is not None:
                mask = np.isin(wavenumber, res.userkws.get("x"))
                fit_curve[mask] = res.best_fit
        data["Fit_Curve"] = fit_curve

    pd.DataFrame(data).to_csv(filepath, index=False)


def export_spectrum_data(filepath: Path, wavenumber: np.ndarray, spectra: np.ndarray) -> None:
    """全ピクセルの処理後波形を (Wavenumber, Pixel_0, ...) 形式でCSV保存する"""
    filepath.parent.mkdir(parents=True, exist_ok=True)
    spectra = np.atleast_2d(spectra)
    cols = {"Wavenumber": wavenumber}
    for i in range(len(spectra)):
        cols[f"Pixel_{i}"] = spectra[i]
    pd.DataFrame(cols).to_csv(filepath, index=False)


def plot_all_spectra_overlay(
    filepath: Path,
    wavenumber: np.ndarray,
    spectra: np.ndarray,
    title: str = "All Processed Spectra Overlay",
    mask_range: tuple[float, float] = (1100, 3000),
    max_percentile: float = 99.0,
) -> None:
    """全ピクセルの処理後スペクトルを重ね合わせてプロット保存する"""
    filepath.parent.mkdir(parents=True, exist_ok=True)
    spectra = np.atleast_2d(spectra)
    mask = (wavenumber >= mask_range[0]) & (wavenumber <= mask_range[1])
    wn_plot = wavenumber[mask]
    spectra_plot = spectra[:, mask]

    plt.figure(figsize=(12, 6))
    for spec in spectra_plot:
        plt.plot(wn_plot, spec, alpha=0.3, linewidth=0.8)

    if spectra_plot.size > 0:
        y_max = np.percentile(np.max(spectra_plot, axis=1), max_percentile)
        y_min = np.percentile(spectra_plot, 0.5)
        y_range = y_max - y_min if y_max > y_min else 1.0
        plt.ylim(y_min - y_range * 0.05, y_max + y_range * 0.15)

    plt.xlabel(r"Wavenumber ($cm^{-1}$)")
    plt.ylabel("Intensity (a.u.)")
    plt.title(title)
    plt.xlim(mask_range)
    plt.grid(True, linestyle="--", alpha=0.5)
    plt.tight_layout()
    plt.savefig(filepath, dpi=150)
    plt.close()


def save_ratio_map_figure(
    filepath: Path,
    x: np.ndarray,
    y: np.ndarray,
    values: np.ndarray,
    title: str = "2D/G Ratio Map",
    cmap: str = "cividis",
    vmin: float | None = 0.0,
    vmax: float | None = 2.0,
) -> None:
    """2次元カラーマップを生成して保存する (欠損グリッドも自動でピボット補完)"""
    filepath.parent.mkdir(parents=True, exist_ok=True)
    plt.rcParams.update({**PLOT_STYLE, "font.size": 16})

    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    values = np.asarray(values, dtype=float)
    x_unique, y_unique = np.unique(x), np.unique(y)

    if len(x_unique) * len(y_unique) == len(values):
        Z = values.reshape(len(y_unique), len(x_unique))
        x_grid, y_grid = x_unique, y_unique
    else:
        logger.warning(
            f"Grid mismatch: {len(x_unique)} (x) * {len(y_unique)} (y) != {len(values)} points. "
            "Gridding by coordinates (missing points -> NaN)."
        )
        pivot = pd.DataFrame({"X": np.round(x, 3), "Y": np.round(y, 3), "Z": values}).pivot_table(
            index="Y", columns="X", values="Z", aggfunc="mean", dropna=False
        )
        x_grid, y_grid, Z = pivot.columns.to_numpy(), pivot.index.to_numpy(), pivot.to_numpy()

    cmap_obj = colors.ListedColormap(plt.get_cmap(cmap)(np.linspace(0, 1, 256)))
    cmap_obj.set_bad("k")

    mesh = plt.pcolormesh(
        x_grid, y_grid, np.ma.masked_invalid(Z), cmap=cmap_obj, vmin=vmin, vmax=vmax, shading="auto"
    )
    plt.colorbar(mesh, label=title)
    plt.title(title)
    plt.tight_layout()
    plt.savefig(filepath, bbox_inches="tight")
    plt.close()


def spectra_draw(
    wavenumber: np.ndarray,
    spectra: np.ndarray,
    idx: list[int] | np.ndarray,
    output_dir: Path,
    output_prefix: str,
    spectra_ref: np.ndarray | None = None,
    fit_results_dict: dict[int, dict] | None = None,
    mask_range: tuple[float, float] = (0, 3000),
    show_peak_ranges: bool = False,
) -> None:
    """指定インデックスのピクセルについてデバッグ用スペクトル画像を出力する"""
    output_dir.mkdir(parents=True, exist_ok=True)
    spectra = np.atleast_2d(spectra)
    plt.rcParams.update({**PLOT_STYLE, "font.size": 10, "savefig.bbox": "tight"})

    mask = (wavenumber >= mask_range[0]) & (wavenumber <= mask_range[1])
    wn_plot = wavenumber[mask]

    for i in idx:
        plt.figure(figsize=(12, 6))
        plt.plot(wn_plot, spectra[i, mask], color="#000000", alpha=0.3, label="Raw Data", zorder=1)

        if spectra_ref is not None:
            plt.plot(wn_plot, spectra_ref[i, mask], color="#0072b2", alpha=0.4, label="Reference")

        if fit_results_dict is not None and i in fit_results_dict:
            for key, res in fit_results_dict[i].items():
                x_fit = res.userkws["x"]
                plt.plot(x_fit, res.best_fit, label=f"Best Fit ({key.upper()})", linewidth=2, zorder=3)
                comps = res.eval_components()
                bkg_val = comps.get("bkg_", np.zeros_like(x_fit))
                plt.plot(x_fit, bkg_val, "k:", alpha=0.5, label="Local Bkg", zorder=2)
                for comp_name, comp_val in comps.items():
                    if "p" in comp_name:
                        plt.fill_between(
                            x_fit, bkg_val, comp_val + bkg_val, color="#56b4e9", alpha=0.2, zorder=2
                        )

        if show_peak_ranges:
            for rng, color, lbl in zip(
                [G_PEAK_RANGE, D_PEAK_RANGE, TWO_D_PEAK_RANGE],
                ["#009e73", "#cc79a7", "#e69f00"],
                ["G-range", "D-range", "2D-range"],
            ):
                plt.axvspan(*rng, color=color, alpha=0.05, label=lbl)
                m = (wavenumber >= rng[0]) & (wavenumber <= rng[1])
                if m.any():
                    peak_wn = wavenumber[m][np.argmax(spectra[i, m])]
                    plt.axvline(peak_wn, color="#d55e00", linestyle=":", alpha=0.6)
                    plt.text(
                        peak_wn, plt.ylim()[1] * 0.9, f"{peak_wn:.1f}",
                        color="#d55e00", rotation=90, verticalalignment="top",
                    )

        plt.title(f"Spectral Analysis - Pixel {i}")
        plt.xlabel(r"Wavenumber ($cm^{-1}$)")
        plt.ylabel("Intensity (a.u.)")
        plt.grid(True, which="both", linestyle="--", alpha=0.5)
        plt.xlim(mask_range)
        plt.savefig(output_dir / f"{output_prefix}_pixel_{i}.png", dpi=150, bbox_inches="tight")
        plt.close()


In [5]:
def analyze_single_spectrum(filepath: Path) -> None:
    """1D単一スペクトルデータの前処理・フィッティング・CSV保存を実行する"""
    save_dir = OUTPUT_DIR / filepath.relative_to(DATA_DIR).parent
    save_dir.mkdir(parents=True, exist_ok=True)

    wavenumber, raw_spectrum = load_single_ascii(filepath)
    processed_spectrum = preprocess_baseline(raw_spectrum, wavenumber, target_range=(1100, 3000))[0]

    df_quant, fit_results_dict = fit_all_spectra(wavenumber, processed_spectrum)
    df_quant.insert(0, "File", filepath.name)

    export_single_spectrum_data(
        save_dir / f"{filepath.stem}_processed.csv",
        wavenumber,
        raw_spectrum,
        processed_spectrum,
        fit_results_dict.get(0),
    )
    df_quant.to_csv(save_dir / f"{filepath.stem}_quantitative.csv", index=False)

    r = df_quant.iloc[0]
    logger.info(f"D/G Ratio: {r['D_G_Ratio']:.4f}, 2D/G Ratio: {r['2D_G_Ratio']:.4f}")
    logger.info(f"2D FWHM: {r['2D_FWHM']:.1f} cm-1, 2D Center: {r['2D_Center']:.1f} cm-1")
    logger.info(f"Single spectrum analysis completed: {filepath.name}")


def analyze_graphene_spectra(
    filepath: Path,
    d_g_ratio_min: float | None = 0.0,
    d_g_ratio_max: float | None = None,
    two_d_g_ratio_min: float | None = 0.0,
    two_d_g_ratio_max: float | None = None,
    target_idx: list[int] | None = None,
    ig_threshold: float = 5.0,
) -> None:
    """グラフェンラマンデータの自動判定（1D/2D）・前処理・フィッティング・マップ/CSV出力を実行する"""
    with open(filepath, "r") as f:
        if len(f.readline().strip().split()) == 2:
            logger.info(f"Detected 1D spectrum: {filepath.name}. Delegating to analyze_single_spectrum.")
            analyze_single_spectrum(filepath)
            return

    save_dir = OUTPUT_DIR / filepath.relative_to(DATA_DIR).parent
    debug_dir = save_dir / "debug"
    if debug_dir.exists():
        for ext in ("*.png", "*.csv"):
            for file in debug_dir.glob(ext):
                file.unlink()

    x, y, wavenumber, raw_spectra = load_mapping_ascii(filepath)
    idx = target_idx if target_idx is not None else np.random.choice(len(raw_spectra), min(5, len(raw_spectra)), replace=False)

    # 1. Siピーク波数シフト補正
    wn_corrected, _ = correct_wavenumber_shift(raw_spectra, wavenumber)
    spectra_draw(wavenumber, wn_corrected, idx, debug_dir, "wavenumber_corrected", spectra_ref=raw_spectra)

    # 2. ALSベースライン補正
    processed_spectra = preprocess_baseline(wn_corrected, wavenumber, target_range=(1100, 3000))
    spectra_draw(wavenumber, processed_spectra, idx, debug_dir, "baseline", spectra_ref=wn_corrected)

    # 3. ピークフィッティング & I_G 閾値マスク適用
    df_results, fit_results_dict = fit_all_spectra(wavenumber, processed_spectra, x=x, y=y)
    masked_cols = ["D_G_Ratio", "2D_G_Ratio", "2D_FWHM", "D_Center", "G_Center", "2D_Center", "2D_RedChi"]
    df_results.loc[df_results["G_Intensity"] <= ig_threshold, masked_cols] = np.nan

    # 4. CSV出力
    save_dir.mkdir(parents=True, exist_ok=True)
    df_results.to_csv(save_dir / "quantitative_results.csv", index=False)
    export_spectrum_data(save_dir / "all_processed_spectra.csv", wavenumber, processed_spectra)
    for i in idx:
        export_single_spectrum_data(
            debug_dir / f"representative_spectrum_{i}.csv",
            wavenumber,
            raw_spectra[i],
            processed_spectra[i],
            fit_results_dict.get(int(i)),
        )

    # 5. マップ・グラフ出力
    plot_all_spectra_overlay(
        save_dir / "all_processed_spectra.png",
        wavenumber,
        processed_spectra,
        title=f"All Processed Spectra - {filepath.stem}",
    )
    ratio_2dg = df_results["2D_G_Ratio"].to_numpy()
    save_ratio_map_figure(
        save_dir / "2d_g_ratio_map.png", x, y, ratio_2dg, title="2D/G Ratio Map",
        vmin=float(np.nanpercentile(ratio_2dg, 25)) if two_d_g_ratio_min is None else two_d_g_ratio_min,
        vmax=float(np.nanpercentile(ratio_2dg, 75)) if two_d_g_ratio_max is None else two_d_g_ratio_max,
    )
    ratio_gd = df_results["D_G_Ratio"].to_numpy()
    save_ratio_map_figure(
        save_dir / "d_g_ratio_map.png", x, y, ratio_gd, title="D/G Ratio Map",
        vmin=float(np.nanpercentile(ratio_gd, 25)) if d_g_ratio_min is None else d_g_ratio_min,
        vmax=float(np.nanpercentile(ratio_gd, 75)) if d_g_ratio_max is None else d_g_ratio_max,
    )
    spectra_draw(
        wavenumber, processed_spectra, idx, debug_dir, "final_fit",
        fit_results_dict=fit_results_dict, mask_range=(1100, 3000), show_peak_ranges=True,
    )

    # 6. サマリーログ
    fwhm_2d = df_results["2D_FWHM"].to_numpy()
    logger.info(f"D/G Ratio - min: {np.nanmin(ratio_gd):.3f}, max: {np.nanmax(ratio_gd):.3f}, median: {np.nanmedian(ratio_gd):.3f}")
    logger.info(f"2D/G Ratio - min: {np.nanmin(ratio_2dg):.3f}, max: {np.nanmax(ratio_2dg):.3f}, median: {np.nanmedian(ratio_2dg):.3f}")
    logger.info(f"2D FWHM (cm-1) - min: {np.nanmin(fwhm_2d):.1f}, max: {np.nanmax(fwhm_2d):.1f}, median: {np.nanmedian(fwhm_2d):.1f}")
    logger.info(
        f"D Center (cm-1) - median: {np.nanmedian(df_results['D_Center']):.1f}, "
        f"G Center (cm-1) - median: {np.nanmedian(df_results['G_Center']):.1f}, "
        f"2D Center (cm-1) - median: {np.nanmedian(df_results['2D_Center']):.1f}"
    )
    logger.info(f"Finished processing {filepath.name}")


In [6]:
D_G_RATIO_MIN = 0
D_G_RATIO_MAX = 0.3
TWO_D_G_RATIO_MIN = 0
TWO_D_G_RATIO_MAX = 2

source_dirs = [
    # DATA_DIR / "spectra_mapping" / "S1" / "S1_sub-si_hole-100_D03_acq-5x3_x100.txt",
    # DATA_DIR / "spectra_mapping" / "S2" / "S2_sub-si_hole-100_D03_acq-5x3_x100.txt",
    # DATA_DIR / "spectra_mapping" / "S4" / "S4_sub-si_hole-100_D03_acq-5x3_x100.txt",
    # DATA_DIR / "spectra_mapping" / "S5" / "S5_sub-si_hole-100_D03_acq-5x3_x100.txt",
    # DATA_DIR / "spectra_mapping" / "S7" / "S7_sub-si_hole-100_D03_acq-5x3_x100.txt",
    # DATA_DIR / "spectra_mapping" / "S8" / "S8_sub-si_hole-100_D03_acq-5x3_x100.txt",
    # DATA_DIR / "spectra_mapping" / "S11" / "SID-11_sub-si&hbn_aneal-1h_growth-2h_grating-100_acq-5x3.txt",
    # DATA_DIR / "spectra_mapping" / "S11b" / "S11b_sub-si_hole-100_D03_acq-5x3_x100.txt",
    # DATA_DIR / "spectra_mapping" / "S11c" / "S11c_hole-100_x50_acq1sx1_grating-100_D03.txt",
    # DATA_DIR / "spectra_mapping" / "S11d" / "S11d_hole-100_x100_acq5sx1_grating-100_D03.txt",
    # DATA_DIR / "spectra_mapping" / "S12" / "SID-12_sub-si_aneal-1h_growth-2h_grating-100_acq-5x3_scale-x50.txt",
    # DATA_DIR / "spectra_mapping" / "S13" / "SID-13_sub-si_grating-100_acq-5x3_scale-x100.txt",
    # DATA_DIR / "spectra_mapping" / "S14" / "SID-14_grating-100_acq5x1_D03_hole-100_scale-x100.txt",
    # DATA_DIR / "spectra_mapping" / "S15" / "SID-15_grating-100_acq5x3_D03_hole-100_scale-x100.txt",
    # DATA_DIR / "spectra_mapping" / "S19" / "S19_hole-100_x100_acq5sx3_grating-100_D03.txt",
    # DATA_DIR / "spectra_mapping" / "S20a" / "S20a_hole-100_x100_acq5sx3_grating-100_D03.txt",
    # DATA_DIR / "spectra_mapping" / "S24b" / "S24b_hole-100_x100_acq5sx3_grating-100_D03.txt",
    # DATA_DIR / "spectra_mapping" / "S34" / "S34_x100_D03_hole-100_acq5sx3.txt",
    # DATA_DIR / "spectra_mapping" / "S35" / "s35_d03_acq5sx3_x50.txt",
    # DATA_DIR / "spectra_mapping" / "S36" / "s36_d03_acq5sx3_x100.txt",
    # DATA_DIR / "spectra_mapping" / "S37" / "s37_D03_x100_acq5sx3.txt",
    # DATA_DIR / "spectra_mapping" / "S43" / "s43_D03_x100_acq5sx3.txt",
    # DATA_DIR / "spectra_mapping" / "S48" / "S48_x100_D03_hole-100_acq5sx3.txt",
    # DATA_DIR / "spectra_mapping" / "S49" / "S49_x100_D03_hole-100_acq5sx3 2.txt",
    # DATA_DIR / "spectra_mapping" / "S50" / "S50_x100_D03_hole-100_acq5sx3.txt",
    # DATA_DIR / "spectra_mapping" / "S51" / "S51_x100_D03_hole-100_acq5sx3.txt",
    # DATA_DIR / "spectra_mapping" / "D31" / "d31_d--_hole-100_x100_acq1sx1.txt",
    # DATA_DIR / "spectra_mapping" / "D33" / "d33_d--_hole-100_x100_acq1sx1.txt",
    # DATA_DIR / "spectra_mapping" / "D35" / "d35_d--_hole-100_x100_acq5sx1.txt",
    # DATA_DIR / "spectra_mapping" / "S60" / "S60_D--_hole-100_x100_acq5sx1.txt",
    # DATA_DIR / "spectra_mapping" / "S61" / "S61_D--_hole-100_x100_acq5sx1.txt",
    # DATA_DIR / "spectra_mapping" / "S62" / "S62_D--_hole-100_x100_acq5sx1.txt",
    # DATA_DIR / "spectra_mapping" / "S63" / "S63_D--_hole-100_x100_acq5sx1.txt",
    # DATA_DIR / "spectra_mapping" / "D38" / "D38_D--_hole-100_x100_acq5sx1.txt",
    # DATA_DIR / "spectra_mapping" / "D39" / "D39_D--_hole-100_x100_acq5sx1.txt",
    # DATA_DIR / "spectra_mapping" / "D40" / "d40_d---_hole-100_x100_acq5sx1.txt",
    # DATA_DIR / "spectra_mapping" / "S64" / "s64_d---_hole-100_x100_acq5sx1.txt",
    # DATA_DIR / "spectra_mapping" / "S65" / "s65_d---_hole-100_x100_acq5sx1.txt",
    # DATA_DIR / "spectra_mapping" / "S66" / "s66_d---_hole-100_x100_acq5sx1.txt",
    # DATA_DIR / "spectra_mapping" / "S67" / "S67_d---_hole-100_x100_acq5sx1.txt",
    # DATA_DIR / "spectra_mapping" / "D42" / "D42_d---_hole-100_x100_acq5sx1.txt",
    # DATA_DIR / "spectra_mapping" / "S68" / "S68_d---_hole-100_x100_acq5sx1.txt",
    # DATA_DIR / "spectra_mapping" / "S69" / "S69_d---_hole-100_x100_acq5sx1.txt",
    # DATA_DIR / "spectra_mapping" / "S70" / "S70_d---_hole-100_x100_acq5sx3.txt",
    # DATA_DIR / "spectra_mapping" / "S71" / "S71_d---_hole-100_x100_acq5sx3.txt",
    # DATA_DIR / "spectra_mapping" / "S72" / "S72_d---_hole-100_x100_acq5sx3.txt",
    # DATA_DIR / "spectra_mapping" / "D43" / "D43_d---_hole-100_x100_acq5sx1.txt",
    # DATA_DIR / "spectra_mapping" / "D44" / "D44_d---_hole-100_x100_acq5sx1.txt",
    # DATA_DIR / "spectra_mapping" / "D45" / "D45_d---_hole-100_x100_acq1sx1.txt",
    # DATA_DIR / "spectra_mapping" / "D45" / "D45_d---_hole-100_x100_acq5sx1.txt",
    # DATA_DIR / "spectra_mapping" / "S73" / "S73_d---_hole-100_x100_acq5sx3.txt",
    DATA_DIR / "spectra_mapping" / "D51" / "D51_d---_hole-100_grad-100_x100_acq5sx3.txt",
]

for src in source_dirs:
    analyze_graphene_spectra(
        src,
        d_g_ratio_min=D_G_RATIO_MIN,
        d_g_ratio_max=D_G_RATIO_MAX,
        two_d_g_ratio_min=TWO_D_G_RATIO_MIN,
        two_d_g_ratio_max=TWO_D_G_RATIO_MAX,
    )


2026-11-06 15:36:59,200 - __main__ - INFO - Fitting all pixels using joblib multiprocessing...
[Parallel(n_jobs=-1)]: Using backend LokyBackend with 8 concurrent workers.
[Parallel(n_jobs=-1)]: Done   2 tasks      | elapsed:    2.0s
[Parallel(n_jobs=-1)]: Done  56 tasks      | elapsed:    5.3s
[Parallel(n_jobs=-1)]: Done 146 tasks      | elapsed:    8.4s
[Parallel(n_jobs=-1)]: Done 272 tasks      | elapsed:   12.8s
[Parallel(n_jobs=-1)]: Done 483 out of 483 | elapsed:   16.6s finished
2026-11-06 15:37:17,629 - __main__ - INFO - D/G Ratio - min: 0.000, max: 24.927, median: 1.123
2026-11-06 15:37:17,630 - __main__ - INFO - 2D/G Ratio - min: 0.000, max: 31.095, median: 1.215
2026-11-06 15:37:17,630 - __main__ - INFO - 2D FWHM (cm-1) - min: 6.0, max: 200.0, median: 41.9
2026-11-06 15:37:17,631 - __main__ - INFO - D Center (cm-1) - median: 1364.9, G Center (cm-1) - median: 1562.6, 2D Center (cm-1) - median: 2637.3
2026-11-06 15:37:17,631 - __main__ - INFO - Finished processing D51_d---_hole

In [7]:
def plot_raman_data(x: np.ndarray, y: np.ndarray, wavenumber: np.ndarray, spectra: np.ndarray, peak: float, width: float) -> None:
    """指定波数(peak ± width/2)における平均スペクトルと最大強度マッピングを表示する"""
    range_min, range_max = peak - width / 2, peak + width / 2

    fig, ax = plt.subplots(figsize=(4.0, 3.2))
    ax.plot(wavenumber, np.mean(spectra, axis=0), color="#000000", linewidth=1)
    ax.axvspan(range_min, range_max, color="#56b4e9", alpha=0.2, label="Target Range")
    ax.set_xlabel("Raman Shift (cm⁻¹)", fontsize=16)
    ax.set_ylabel("Intensity (a.u.)", fontsize=16)
    ax.tick_params(axis="both", labelsize=16)
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=16)
    plt.tight_layout()
    plt.show()

    mask = (wavenumber >= range_min) & (wavenumber <= range_max)
    if not np.any(mask):
        print(f"Warning: No data points found in range {range_min} - {range_max}")
        return

    pivot = pd.DataFrame({"X": y, "Y": x, "Intensity": np.max(spectra[:, mask], axis=1)}).pivot(
        index="Y", columns="X", values="Intensity"
    )
    fig, ax = plt.subplots(figsize=(4.0, 3.2))
    sc = ax.pcolormesh(pivot.columns.to_numpy(), pivot.index.to_numpy(), pivot.to_numpy(), shading="auto", cmap="viridis")
    plt.colorbar(sc, label="Peak Max Intensity (a.u.)", shrink=0.6, aspect=20, pad=0.05)
    ax.add_artist(ScaleBar(1, "um", length_fraction=0.2, location="lower right", box_alpha=0, color="white", font_properties={"size": 12}))
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_aspect("equal")
    ax.invert_yaxis()
    plt.tight_layout()
    plt.show()
